# 06: Full Factorial Execution (960 Runs)

Executes the complete design: 2 datasets x 3 classifiers x
4 imbalance ratios x 4 minimization levels x 2 minimization types x
5 seeds = 960 run-seed combinations. Each run follows the pipeline
imbalance -> minimization -> split -> train -> metrics on test
(experiment_functions.py) with per-run checkpointing to
results_checkpoint.csv and offline W&B logging.

Primary output: results_master.csv, the flat 960-row analytical table
from which all results derive.

In [3]:
import pandas as pd
import numpy as np
import os, time
import wandb

from manipulation_functions import *
from experiment_functions import *

BUCKET = "osilesi-dissertation-data-2026"
datasets = {
    "adult":  pd.read_csv(f"s3://{BUCKET}/processed/adult_final_pruned.csv"),
    "health": pd.read_csv(f"s3://{BUCKET}/processed/health_final_pruned.csv"),
}
wandb.login()

True

In [4]:
import re

def sanitize_columns(df):
    df = df.copy()
    df.columns = [re.sub(r"[\[\]<>]", "_", str(c)) for c in df.columns]
    assert df.columns.is_unique, "sanitization created duplicate names"
    return df

datasets = {k: sanitize_columns(v) for k, v in datasets.items()}

## Design grid

The full 960-condition enumeration as an auditable dataframe, exported
to design_grid.csv (/results). run_id is the row index under this exact
factor ordering and is the join key for the checkpoint and master table.
Seeds fixed at [11, 22, 33, 44, 55] as part of the reproducibility
specification.

Note: minimization type is crossed, not nested, so the 100% level
appears under both horizontal and vertical labels; the two are identical
operations at 100% and produce identical metrics at the same seed
(verified). The duplication is by design, keeping every imbalance x
minimization cell balanced at exactly 60 observations, which underpins
the equal-cell-sizes robustness argument.

In [5]:
from itertools import product

grid = pd.DataFrame(
    product(
        ["adult", "health"],                                   # 2 datasets
        ["logistic_regression", "random_forest", "xgboost"],   # 3 classifiers
        ["50/50", "80/20", "90/10", "95/5"],                   # 4 imbalance
        ["100", "75", "50", "25"],                             # 4 minimization
        ["horizontal", "vertical"],                            # 2 types
        [11, 22, 33, 44, 55],                                  # 5 seeds
    ),
    columns=["dataset", "classifier", "imbalance",
             "min_level", "min_type", "seed"],
)
grid["run_id"] = grid.index
print(len(grid))   # must print 960

960


## Checkpointed execution loop

Every result is appended to results_checkpoint.csv immediately;
completed runs are skipped on restart. load_done() is hardened against
the failure modes observed during this batch: schema-aware read (mixed
row widths), stray re-appended header rows, type-coerced run_ids,
interrupted-write fragments, and error-status rows (excluded from the
done set so failed conditions re-execute). The loop is therefore safe
to interrupt and resume at any point.

In [7]:
CHECKPOINT = "results_checkpoint.csv"

COLS = ["dataset", "classifier", "imbalance", "min_level", "min_type",
        "seed", "accuracy", "minority_recall", "macro_f1",
        "balanced_accuracy", "mcc", "equal_opportunity_diff", "fnr_diff",
        "n_test", "n_female_pos_test", "status", "run_id", "runtime_sec"]

def load_done():
    if os.path.exists(CHECKPOINT):
        done = pd.read_csv(CHECKPOINT, header=None, names=COLS,
                           engine="python")
        done = done[done["dataset"].isin(["adult", "health"])]
        done = done[done["status"] == "ok"]
        done["run_id"] = pd.to_numeric(done["run_id"], errors="coerce")
        done = done.dropna(subset=["run_id"])
        done["run_id"] = done["run_id"].astype(int)
        done = done.drop_duplicates(subset="run_id", keep="first")
        return done, set(done["run_id"])
    return pd.DataFrame(), set()

done_df, done_ids = load_done()
print(f"Resuming: {len(done_ids)} of {len(grid)} complete")

t0 = time.time()
for _, row in grid.iterrows():
    if row["run_id"] in done_ids:
        continue

    r = run_experiment(
        datasets[row["dataset"]], row["dataset"], row["classifier"],
        row["imbalance"], row["min_level"], row["min_type"],
        int(row["seed"]),
    )
    r["run_id"] = row["run_id"]
    r["runtime_sec"] = round(time.time() - t0, 1)

    pd.DataFrame([r]).to_csv(
        CHECKPOINT, mode="a", index=False,
        header=not os.path.exists(CHECKPOINT),
    )

    if row["run_id"] % 25 == 0:
        elapsed = (time.time() - t0) / 60
        print(f"run {row['run_id']:>3} | {elapsed:5.1f} min elapsed")

Resuming: 960 of 960 complete


## Completeness and integrity verification

The checkpoint is a historical log and deliberately retains error rows
as provenance; the clean analytical subset is unique status=ok rows.
Six checks: (1) all 960 designed run_ids completed successfully, none
missing; (2) historical error rows counted and documented; (3) zero
missing metric values; (4) all 16 imbalance x minimization cells at
exactly n = 60; (5) fairness metrics computable in every run
(min female positives > 0); (6) directional sanity of pooled means
against pilot-established theory.

In [10]:
import pandas as pd

COLS = ["dataset", "classifier", "imbalance", "min_level", "min_type",
        "seed", "accuracy", "minority_recall", "macro_f1",
        "balanced_accuracy", "mcc", "equal_opportunity_diff", "fnr_diff",
        "n_test", "n_female_pos_test", "status", "run_id", "runtime_sec"]

raw = pd.read_csv(CHECKPOINT, header=None, names=COLS, engine="python")
raw = raw[raw["dataset"].isin(["adult", "health"])]   # drop header-text rows

# Clean subset: successful runs only, deduplicated
res = raw[raw["status"] == "ok"].copy()
res["run_id"] = pd.to_numeric(res["run_id"], errors="coerce")
res = res.dropna(subset=["run_id"])
res["run_id"] = res["run_id"].astype(int)
res = res.drop_duplicates(subset="run_id", keep="first")

metric_cols = ["accuracy", "minority_recall", "macro_f1",
               "balanced_accuracy", "mcc",
               "equal_opportunity_diff", "fnr_diff"]
for c in metric_cols + ["n_test", "n_female_pos_test", "seed"]:
    res[c] = pd.to_numeric(res[c], errors="coerce")
res[["n_test", "n_female_pos_test", "seed"]] = \
    res[["n_test", "n_female_pos_test", "seed"]].astype(int)

# 1. Every one of the 960 designed runs completed successfully
assert res["run_id"].nunique() == 960, \
    f"only {res['run_id'].nunique()} unique ok runs"
missing = set(range(960)) - set(res["run_id"])
assert not missing, f"missing run_ids: {sorted(missing)}"

# 2. Historical failures are documented, and none is unresolved
n_err_rows = (raw["status"] != "ok").sum()

# 3. No missing metric values among completed runs
assert res[metric_cols].isna().sum().sum() == 0

# 4. Balanced cells: 60 per imbalance x min_level combination
cell_counts = res.groupby(["imbalance", "min_level"]).size()
print(cell_counts)
assert (cell_counts == 60).all()

# 5. Fairness metrics computable everywhere
print("min female positives in any test set:",
      res["n_female_pos_test"].min())
assert res["n_female_pos_test"].min() > 0

# 6. Sanity: pilot patterns hold at scale
print(res.groupby("imbalance")[["accuracy", "minority_recall", "mcc"]]
         .mean().round(3))

imbalance  min_level
50/50      100          60
           25           60
           50           60
           75           60
80/20      100          60
           25           60
           50           60
           75           60
90/10      100          60
           25           60
           50           60
           75           60
95/5       100          60
           25           60
           50           60
           75           60
dtype: int64
min female positives in any test set: 8
           accuracy  minority_recall    mcc
imbalance                                  
50/50         0.718            0.711  0.437
80/20         0.832            0.321  0.340
90/10         0.909            0.210  0.267
95/5          0.953            0.136  0.213


## Verification: PASSED 2026-09-14

All six checks passed on the completed batch:

1. 960/960 unique successful runs; missing set empty
2. Zero missing metric values across all seven DVs
3. All 16 cells at exactly n = 60
4. Min female positives in any test set = 8 (95/5 x 25% conditions;
   consistent with pilot
5. Pooled means confirm theory: accuracy 0.718 -> 0.953 across
   worsening imbalance while minority recall falls 0.711 -> 0.136
   and MCC 0.437 -> 0.213. Pooled 50/50 accuracy sits below the
   adult-only pilot values because health (readmission) is the
   harder task; dataset difference noted for Supplementary Findings.

## Master table export

results_master.csv: the deduplicated, type-corrected, run_id-sorted
960-row table (all IVs, all seven DVs, diagnostics, export timestamp).
Written to S3 /results, local download, and one off-AWS backup (three
copies, two media). Environment frozen at this state via
pip freeze > requirements.txt; all subsequent sessions install from
this file (never -U) so analysis and the reproducibility audit run on
the exact versions that produced these results.

In [9]:
res = res.sort_values("run_id").reset_index(drop=True)
res["exported_at"] = pd.Timestamp.now().isoformat()
res.to_csv("results_master.csv", index=False)
res.to_csv(f"s3://{BUCKET}/results/results_master.csv", index=False)
print(res.shape)   # (960, 19)

(960, 19)
